In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as f
from pyspark.sql.types import StringType

filepath = "file:///D:/BDA 20/ABD-BDA-LabWork/datasets/0909 Lab/indian_food.csv"

In [3]:
def create_session():
    spark = SparkSession.builder.appName('FoodDishes').getOrCreate()
    return spark

In [4]:
def create_df(spark, filepath):
    df = spark.read.csv(filepath, header = True)
    return df

In [5]:
spark = create_session()
df = create_df(spark, filepath)
df.printSchema()

root
 |-- name: string (nullable = true)
 |-- ingredients: string (nullable = true)
 |-- diet: string (nullable = true)
 |-- prep_time: string (nullable = true)
 |-- cook_time: string (nullable = true)
 |-- flavor_profile: string (nullable = true)
 |-- course: string (nullable = true)
 |-- state: string (nullable = true)



In [6]:
df.show(5, truncate=False)

+--------------+----------------------------------------------------------------------------------------------+----------+---------+---------+--------------+-------+-----------+
|name          |ingredients                                                                                   |diet      |prep_time|cook_time|flavor_profile|course |state      |
+--------------+----------------------------------------------------------------------------------------------+----------+---------+---------+--------------+-------+-----------+
|Balu shahi    |Maida flour, yogurt, oil, sugar                                                               |vegetarian|45       |25       |sweet         |dessert|West Bengal|
|Boondi        |Gram flour, ghee, sugar                                                                       |vegetarian|80       |30       |sweet         |dessert|Rajasthan  |
|Gajar ka halwa|Carrots, milk, sugar, ghee, cashews, raisins                                                  

In [7]:
# 1. Find out how many unique dishes are present. 


total = df.count()
unique= df.select("name").distinct().count()
print("Total rows:", total)
print("Unique dishes:",unique)

Total rows: 255
Unique dishes: 255


In [8]:
# 2. Which state has more dishes? 

# dishes_per_state = df.groupBy("state").count()

# dishes_per_state.orderBy("count", ascending=False).show()

# top_state = dishes_per_state.orderBy("count", ascending=False).first()

# print("State with most dishes:", top_state["state"])
# print("Number of dishes:", top_state["count"])

df.groupBy("state").count().orderBy("count", ascending=False).show()

+---------------+-----+
|          state|count|
+---------------+-----+
|        Gujarat|   35|
|         Punjab|   32|
|    Maharashtra|   30|
|             -1|   24|
|    West Bengal|   24|
|          Assam|   21|
|     Tamil Nadu|   20|
| Andhra Pradesh|   10|
|  Uttar Pradesh|    9|
|         Kerala|    8|
|         Odisha|    7|
|      Karnataka|    6|
|      Rajasthan|    6|
|      Telangana|    5|
|            Goa|    3|
|          Bihar|    3|
| Madhya Pradesh|    2|
|        Manipur|    2|
|Jammu & Kashmir|    2|
|       Nagaland|    1|
+---------------+-----+
only showing top 20 rows



In [9]:
# 3. How many dishes from state Karnataka? 
df.filter(df.state == "Karnataka").groupBy("state").count().show()

+---------+-----+
|    state|count|
+---------+-----+
|Karnataka|    6|
+---------+-----+



In [10]:
# 4. List number of unique regions 
region_map = {
    "Punjab": "North",
    "Haryana": "North",
    "Rajasthan": "North",
    "Uttar Pradesh": "North",
    "Uttarakhand": "North",
    "NCT of Delhi": "North",
    "Jammu & Kashmir": "North",

    "Karnataka": "South",
    "Kerala": "South",
    "Tamil Nadu": "South",
    "Andhra Pradesh": "South",
    "Telangana": "South",

    "West Bengal": "East",
    "Odisha": "East",
    "Bihar": "East",
    "Jharkhand": "East",

    "Assam": "North East",
    "Manipur": "North East",
    "Nagaland": "North East",
    "Tripura": "North East",
    "Meghalaya": "North East",
    "Mizoram": "North East",
    "Arunachal Pradesh": "North East",
    "Sikkim": "North East",

    "Gujarat": "West",
    "Maharashtra": "West",
    "Goa": "West",

    "Madhya Pradesh": "Central",
    "Chhattisgarh": "Central"
}

# Add region using when()
df = df.withColumn(
    "region",
    f.when(f.col("state").isin(
        "Punjab", "Haryana", "Rajasthan", "Uttar Pradesh",
        "Uttarakhand", "NCT of Delhi", "Jammu & Kashmir"
    ), "North")
    .when(f.col("state").isin(
        "Karnataka", "Kerala", "Tamil Nadu",
        "Andhra Pradesh", "Telangana"
    ), "South")
    .when(f.col("state").isin(
        "West Bengal", "Odisha", "Bihar", "Jharkhand"
    ), "East")
    .when(f.col("state").isin(
        "Assam", "Manipur", "Nagaland", "Tripura",
        "Meghalaya", "Mizoram", "Arunachal Pradesh", "Sikkim"
    ), "North East")
    .when(f.col("state").isin(
        "Gujarat", "Maharashtra", "Goa"
    ), "West")
    .when(f.col("state").isin(
        "Madhya Pradesh", "Chhattisgarh"
    ), "Central")
    .otherwise("Unknown (-1)")
)

print("Regions:")
df.select("region").distinct().show()

Regions:
+------------+
|      region|
+------------+
|Unknown (-1)|
|       South|
|     Central|
|        East|
|        West|
|  North East|
|       North|
+------------+



In [13]:
# 5. Count number of dishes from each region. 
print("Dishes per region:")

df.groupBy("region").count().orderBy(f.desc("count")).show()

Dishes per region:
+------------+-----+
|      region|count|
+------------+-----+
|        West|   68|
|       North|   52|
|       South|   49|
|        East|   34|
|  North East|   25|
|Unknown (-1)|   24|
|     Central|    3|
+------------+-----+



In [14]:
# 6. List unique 'flavor_profile' and 'course' 
print("Unique flavor profiles:")
df.select("flavor_profile").distinct().show()

print("Unique courses:")
df.select("course").distinct().show()

Unique flavor profiles:
+--------------+
|flavor_profile|
+--------------+
|            -1|
|         spicy|
|         sweet|
|          sour|
|        bitter|
+--------------+

Unique courses:
+-----------+
|     course|
+-----------+
|    starter|
|    dessert|
|      snack|
|main course|
+-----------+



In [15]:
# 7. Which state has more 'main course'? 
main_course = df.filter(df["course"] == "main course")

result = (
    main_course.groupBy("state")
    .count()
    .orderBy("count", ascending=False)
)

result.show()

top = result.first()

print("State with most main course dishes:", top["state"])
print("Number of dishes:", top["count"])

+---------------+-----+
|          state|count|
+---------------+-----+
|         Punjab|   28|
|     Tamil Nadu|   17|
|          Assam|   15|
|        Gujarat|   12|
|    Maharashtra|   12|
|             -1|    9|
|    West Bengal|    9|
|         Kerala|    5|
|      Karnataka|    4|
|      Rajasthan|    3|
|  Uttar Pradesh|    3|
|          Bihar|    2|
|       Nagaland|    1|
|         Odisha|    1|
| Madhya Pradesh|    1|
|        Manipur|    1|
|Jammu & Kashmir|    1|
|            Goa|    1|
|        Haryana|    1|
|   NCT of Delhi|    1|
+---------------+-----+
only showing top 20 rows

State with most main course dishes: Punjab
Number of dishes: 28


In [23]:
# 8. Give the %of dishes from each region. 
dishes_per_region = (
    df.groupBy("region")
      .count()
      .orderBy(f.desc("count"))
)

total = df.count()

region_percentage = dishes_per_region.withColumn(
    "percentage",f.round((f.col("count") / total) * 100, 4)
)

region_percentage.show()

+------------+-----+----------+
|      region|count|percentage|
+------------+-----+----------+
|        West|   68|   26.6667|
|       North|   52|   20.3922|
|       South|   49|   19.2157|
|        East|   34|   13.3333|
|  North East|   25|    9.8039|
|Unknown (-1)|   24|    9.4118|
|     Central|    3|    1.1765|
+------------+-----+----------+



In [34]:
# 9. List the states which has more dishes from each region. 
from pyspark.sql.window import Window
state_count = df.groupBy("region", "state").count()

window = Window.partitionBy("region").orderBy(f.desc("count"))

top_state = (
    state_count
    .withColumn("rank", f.row_number().over(window))
    .filter(f.col("rank") == 1)
    .drop("rank")
)

top_state.show()

+------------+--------------+-----+
|      region|         state|count|
+------------+--------------+-----+
|     Central|Madhya Pradesh|    2|
|        East|   West Bengal|   24|
|       North|        Punjab|   32|
|  North East|         Assam|   21|
|       South|    Tamil Nadu|   20|
|Unknown (-1)|            -1|   24|
|        West|       Gujarat|   35|
+------------+--------------+-----+

